# Graph Data Analysis

This notebook evaluates the intrinsic properties of the SLN graphs. It computes metrics such as graph density, connected component sizes, and algebraic connectivity, offering insights into the structural evolution of the classrooms. Refer to `docs/07_Data_Analysis.md` for more details.

### Source: GraphDensity.ipynb

In [ ]:
!pip install  dgl -f https://data.dgl.ai/wheels/torch-2.4/repo.html

In [ ]:
import dgl
import networkx as nx
from networkx.linalg.algebraicconnectivity import algebraic_connectivity

def largest_component_algebraic_connectivity(dgl_graph):
    dgl_graph = dgl_graph.cpu()  # Ensure the graph is on the CPU

    # Convert DGL graph to NetworkX graph and make it undirected
    nx_graph = dgl.to_networkx(dgl_graph).to_undirected()

    # Find the largest connected component
    largest_component_nodes = max(nx.connected_components(nx_graph), key=len)
    largest_component = nx_graph.subgraph(largest_component_nodes).copy()

    # Calculate algebraic connectivity for the largest component
    if largest_component.number_of_nodes() > 1:  # Connectivity is undefined for single-node components
        connectivity = algebraic_connectivity(largest_component, method='lanczos')
    else:
        connectivity = 0.0

    return connectivity

# Example usage
bin_file = 'drive/MyDrive/Colab Notebooks/GNN Research/algo004_results/graphs/69/graphs_algo004_0.25p_seed69.bin'

# Load the graph
graphs, _ = dgl.load_graphs(bin_file)
algo_graph = graphs[0]  # Assuming the first graph is the one you need

# Get the algebraic connectivity of the largest connected component
largest_component_connectivity = largest_component_algebraic_connectivity(algo_graph)
print(f"Algebraic connectivity of the largest connected component: {largest_component_connectivity}")


In [ ]:
import dgl
import networkx as nx

def list_connected_components_sizes(dgl_graph):
    dgl_graph = dgl_graph.cpu()  # Ensure the graph is on the CPU

    # Convert DGL graph to NetworkX graph and make it undirected
    nx_graph = dgl.to_networkx(dgl_graph).to_undirected()

    # Get all connected components and their sizes
    component_sizes = [len(c) for c in nx.connected_components(nx_graph)]

    # Sort the sizes in descending order
    sorted_sizes = sorted(component_sizes, reverse=True)

    return sorted_sizes

# Example usage
bin_file = 'drive/MyDrive/Colab Notebooks/GNN Research/algo004_results/graphs/18/graphs_algo004_0.75p_seed18.bin'

# Load the graph
graphs, _ = dgl.load_graphs(bin_file)
algo_graph = graphs[0]  # Assuming the first graph is the one you need

# Get the sizes of connected components in descending order
connected_component_sizes = list_connected_components_sizes(algo_graph)
print(f"Connected component sizes (descending order): {connected_component_sizes}")


# Average Degree Calculation

In [ ]:
ls drive/MyDrive/Colab\ Notebooks/GNN\ Research/Graph\ Neural\ Network/

In [ ]:
import os
import pandas as pd
import numpy as np
import dgl
import networkx as nx

# Define the datasets, percents, and seeds
datasets = ['algo004', 'comp', 'ml', 'virtualshakespeare']  # adjust as needed
percents = [0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75]  # adjust as needed
seeds = [18, 61, 53, 29, 69, 42, 2, 21, 78, 99]

# Base directory path
base_dir = 'drive/MyDrive/Colab Notebooks/GNN Research/Graph Neural Network'

results = []

for ds in datasets:
    for p in percents:
        avg_degrees = []
        for seed in seeds:
            # Construct the path to the graph file:
            # Example:
            # drive/MyDrive/Colab Notebooks/GNN Research/Graph Neural Network/algo004_results/graphs/69/graphs_algo004_0.05p_seed69.bin
            ds_folder = f'{ds}_results'
            graphs_folder = 'graphs'
            filename = f'graphs_{ds}_{p}p_seed{seed}.bin'
            bin_file = os.path.join(base_dir, ds_folder, graphs_folder, str(seed), filename)

            # Check if the file exists
            if not os.path.isfile(bin_file):
                print(f"File not found: {bin_file}, skipping this seed.")
                continue

            # Load the DGL graph
            graphs, _ = dgl.load_graphs(bin_file)
            g = graphs[0]  # Assuming a single graph per file

            # Convert to NetworkX (undirected)
            nx_g = dgl.to_networkx(g).to_undirected()

            # Compute total nodes and edges
            total_num_nodes = nx_g.number_of_nodes()
            total_num_edges = nx_g.number_of_edges()

            # Average degree = (2 * total_num_edges) / total_num_nodes
            if total_num_nodes > 0:
                avg_degree = (2 * total_num_edges) / total_num_nodes
            else:
                avg_degree = 0.0

            avg_degrees.append(avg_degree)

        # If we got at least one seed's data
        if len(avg_degrees) > 0:
            mean_degree_over_seeds = np.mean(avg_degrees)
        else:
            mean_degree_over_seeds = np.nan  # If no data found for this config

        results.append({
            'Dataset': ds,
            'Percent': p,
            'Average Degree': mean_degree_over_seeds
        })

# Convert results to DataFrame and save
results_df = pd.DataFrame(results)
output_csv = os.path.join(base_dir, 'average_degrees.csv')
results_df.to_csv(output_csv, index=False)
print(f"Average degree results saved to {output_csv}")


# Graph Density Calculation

In [ ]:
import os
import pandas as pd
import numpy as np
import dgl
import networkx as nx

datasets = ['algo004', 'comp', 'ml', 'virtualshakespeare']
percents = [0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75]
seeds = [18, 61, 53, 29, 69, 42, 2, 21, 78, 99]

base_dir = 'drive/MyDrive/Colab Notebooks/GNN Research/Graph Neural Network'

results = []

for ds in datasets:
    for p in percents:
        densities = []
        for seed in seeds:
            ds_folder = f'{ds}_results'
            graphs_folder = 'graphs'
            filename = f'graphs_{ds}_{p}p_seed{seed}.bin'
            bin_file = os.path.join(base_dir, ds_folder, graphs_folder, str(seed), filename)

            if not os.path.isfile(bin_file):
                print(f"File not found: {bin_file}, skipping this seed.")
                continue

            # Load the DGL graph
            graphs, _ = dgl.load_graphs(bin_file)
            g = graphs[0]

            # Convert to NetworkX
            nx_g = dgl.to_networkx(g).to_undirected()

            # Compute total nodes and edges
            total_num_nodes = nx_g.number_of_nodes()
            total_num_edges = nx_g.number_of_edges()

            # Calculate density
            if total_num_nodes > 1:
                density = total_num_edges / (total_num_nodes * (total_num_nodes - 1) / 2)
            else:
                density = 0.0
            densities.append(density)

        # Average over seeds
        if len(densities) > 0:
            mean_density = np.mean(densities)
        else:
            mean_density = np.nan

        results.append({
            'Dataset': ds,
            'Percent': p,
            'Density': mean_density
        })

# Save results to CSV
results_df = pd.DataFrame(results)
output_csv = os.path.join(base_dir, 'graph_densities.csv')
results_df.to_csv(output_csv, index=False)
print(f"Graph densities saved to {output_csv}")


# GED

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import dgl
import networkx as nx
from networkx.algorithms import similarity

def induce_subgraph(nx_graph, target_num_nodes):
    """Randomly pick 'target_num_nodes' from nx_graph and return the induced subgraph."""
    all_nodes = list(nx_graph.nodes())
    if target_num_nodes >= len(all_nodes):
        return nx_graph.copy()
    sampled_nodes = random.sample(all_nodes, target_num_nodes)
    return nx_graph.subgraph(sampled_nodes).copy()

# Example: Compare ml vs. virtualshakespeare
pairs_to_compare = [('ml', 'virtualshakespeare')]
percents = [0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75]
seeds = [18, 61, 53, 29, 69, 42, 2, 21, 78, 99]

base_dir = 'drive/MyDrive/Colab Notebooks/GNN Research/Graph Neural Network'
results = []

for ds1, ds2 in pairs_to_compare:
    print(f"Comparing {ds1} vs. {ds2}")
    for p in percents:
        ged_per_seed = []

        for seed in seeds:
            # Construct file paths
            ds1_path = os.path.join(base_dir, f'{ds1}_results', 'graphs', str(seed),
                                    f'graphs_{ds1}_{p}p_seed{seed}.bin')
            ds2_path = os.path.join(base_dir, f'{ds2}_results', 'graphs', str(seed),
                                    f'graphs_{ds2}_{p}p_seed{seed}.bin')

            if not os.path.isfile(ds1_path):
                print(f"File not found: {ds1_path}, skipping...")
                continue
            if not os.path.isfile(ds2_path):
                print(f"File not found: {ds2_path}, skipping...")
                continue

            # Load
            g1_dgl, _ = dgl.load_graphs(ds1_path)
            g2_dgl, _ = dgl.load_graphs(ds2_path)
            nx_g1_full = dgl.to_networkx(g1_dgl[0]).to_undirected()
            nx_g2_full = dgl.to_networkx(g2_dgl[0]).to_undirected()

            # Identify smaller vs. larger
            n1 = nx_g1_full.number_of_nodes()
            n2 = nx_g2_full.number_of_nodes()
            if n1 <= n2:
                nx_small, nx_large = nx_g1_full, nx_g2_full
                small_size = n1
            else:
                nx_small, nx_large = nx_g2_full, nx_g1_full
                small_size = n2

            # Induce subgraph on the larger graph to match the smaller's node count
            nx_sub = induce_subgraph(nx_large, small_size)

            # Compute GED with a tiny timeout
            try:
                ged_iter = similarity.graph_edit_distance(nx_small, nx_sub, timeout=5)
                if ged_iter is None:
                    # Means it failed to produce an iterator
                    ged_val = float('inf')
                else:
                    ged_val = next(ged_iter, None)
                    if ged_val is None:
                        ged_val = float('inf')
            except Exception as e:
                print(f"Error computing GED: {e}")
                ged_val = float('inf')

            ged_per_seed.append(ged_val)

        # Average for this (p, ds1, ds2)
        if len(ged_per_seed) == 0:
            mean_ged = np.nan
        else:
            valid = [v for v in ged_per_seed if not np.isinf(v)]
            mean_ged = np.mean(valid) if valid else float('inf')

        results.append({
            'Dataset1': ds1,
            'Dataset2': ds2,
            'Percent': p,
            'Mean_GED': mean_ged
        })

# Save results
df = pd.DataFrame(results)
output_csv = os.path.join(base_dir, 'ged_subsample_comparison.csv')
df.to_csv(output_csv, index=False)
print("Saved results to", output_csv)


# Degree Distribution Distance

In [ ]:
import os
import numpy as np
import pandas as pd
import dgl
import networkx as nx
from scipy.stats import wasserstein_distance  # For degree distribution distance

###########################################
# 1) Pair(s) to compare
###########################################
pairs_to_compare = [
    ('ml', 'virtualshakespeare'),
    ('algo004', 'comp'),
    ('ml', 'comp'),
    ('algo004', 'virtualshakespeare'),
    ('ml', 'algo004'),
    ('comp', 'virtualshakespeare')
]

###########################################
# 2) Define percentages and seeds
###########################################
percents = [0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75]
seeds = [18, 61, 53, 29, 69, 42, 2, 21, 78, 99]

base_dir = 'drive/MyDrive/Colab Notebooks/GNN Research/Graph Neural Network'
results = []

###########################################
# 3) Helper function: degree distribution
###########################################
def get_degree_list(dgl_graph):
    """
    Convert a DGL graph to NetworkX (undirected),
    then return a list of degrees of all nodes.
    """
    nx_g = dgl.to_networkx(dgl_graph).to_undirected()
    degrees = [deg for (_, deg) in nx_g.degree()]  # degree() returns (node, degree) pairs
    return degrees

###########################################
# 4) Main loop comparing degree distributions
###########################################
for ds1, ds2 in pairs_to_compare:
    print(f"Comparing {ds1} vs. {ds2}")
    for p in percents:
        dist_per_seed = []  # Store Wasserstein distances for each seed

        for seed in seeds:
            # Build paths for each dataset
            ds1_path = os.path.join(
                base_dir, f'{ds1}_results', 'graphs', str(seed),
                f'graphs_{ds1}_{p}p_seed{seed}.bin'
            )
            ds2_path = os.path.join(
                base_dir, f'{ds2}_results', 'graphs', str(seed),
                f'graphs_{ds2}_{p}p_seed{seed}.bin'
            )

            # Check if files exist
            if not os.path.isfile(ds1_path):
                print(f"File not found: {ds1_path}, skipping.")
                continue
            if not os.path.isfile(ds2_path):
                print(f"File not found: {ds2_path}, skipping.")
                continue

            # Load DGL graphs
            g1_list, _ = dgl.load_graphs(ds1_path)
            g2_list, _ = dgl.load_graphs(ds2_path)
            # Assuming first graph in the file is the main graph
            g1_dgl = g1_list[0]
            g2_dgl = g2_list[0]

            # Compute degree lists
            deg_list1 = get_degree_list(g1_dgl)
            deg_list2 = get_degree_list(g2_dgl)

            # 5) Compute Wasserstein distance (Earth Mover’s Distance)
            #    If either list is empty (e.g., 0 nodes), handle carefully
            if len(deg_list1) == 0 or len(deg_list2) == 0:
                dist = np.nan  # or some sentinel
            else:
                dist = wasserstein_distance(deg_list1, deg_list2)

            dist_per_seed.append(dist)

        # 6) Average across seeds
        if len(dist_per_seed) == 0:
            mean_dist = np.nan
        else:
            # Filter out any NaNs
            valid = [d for d in dist_per_seed if not np.isnan(d)]
            mean_dist = np.mean(valid) if len(valid) > 0 else np.nan

        # Store result
        results.append({
            'Dataset1': ds1,
            'Dataset2': ds2,
            'Percent': p,
            'Mean_Wasserstein_Degree': mean_dist
        })

# 7) Save to CSV
df = pd.DataFrame(results)
output_csv = os.path.join(base_dir, 'degree_dist_comparison.csv')
df.to_csv(output_csv, index=False)
print(f"Saved degree distribution comparison to {output_csv}")


In [ ]:
import os
import pandas as pd
import itertools

# Adjust to your actual CSV path
base_dir = 'drive/MyDrive/Colab Notebooks/GNN Research/Graph Neural Network'
pairwise_csv = os.path.join(base_dir, 'degree_dist_comparison.csv')

# 1) Read the pairwise CSV
df = pd.read_csv(pairwise_csv)

# 2) Create a dictionary for quick lookup: (percent, ds1, ds2) -> mean_wasserstein
#    We'll store ds1 < ds2 in alphabetical order (or any consistent ordering)
pairwise_dict = {}
for _, row in df.iterrows():
    ds1 = row['Dataset1']
    ds2 = row['Dataset2']
    p = row['Percent']
    dist = row['Mean_Wasserstein_Degree']

    # Normalize dataset ordering so (ds1, ds2) always stored with ds1 < ds2
    # This makes lookups easier
    if ds1 > ds2:
        ds1, ds2 = ds2, ds1
    pairwise_dict[(p, ds1, ds2)] = dist

# 3) Suppose we want to use the same set of datasets you already used:
datasets = ['algo004', 'comp', 'ml', 'virtualshakespeare']
percents = [0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75]

# If you only want triplets, use combinations(datasets, 3)
# If you only want quadruplets, use combinations(datasets, 4)
# Or do both in separate loops if needed.
triplets = list(itertools.combinations(datasets, 3))
quadruplets = list(itertools.combinations(datasets, 4))

triplet_results = []
quad_results = []

##########################################
# 4) Calculate triplet distances
##########################################
for p in percents:
    for (dsA, dsB, dsC) in triplets:
        # Sort them to match pairwise_dict
        sorted_three = sorted([dsA, dsB, dsC])
        ds1, ds2, ds3 = sorted_three

        # Retrieve the three pairwise distances if they exist
        d12 = pairwise_dict.get((p, ds1, ds2), float('nan'))
        d23 = pairwise_dict.get((p, ds2, ds3), float('nan'))
        d13 = pairwise_dict.get((p, ds1, ds3), float('nan'))

        # Average them
        valid_dists = [d for d in [d12, d23, d13] if not pd.isna(d)]
        if len(valid_dists) == 3:
            mean_dist = sum(valid_dists) / 3.0
        else:
            mean_dist = float('nan')

        triplet_results.append({
            'Percent': p,
            'Datasets': f"{dsA},{dsB},{dsC}",
            'Triplet_Wasserstein': mean_dist,
            'PairwiseDistances': [d12, d23, d13]
        })

##########################################
# 5) Calculate quadruplet distances
##########################################
for p in percents:
    for (dsA, dsB, dsC, dsD) in quadruplets:
        sorted_four = sorted([dsA, dsB, dsC, dsD])
        ds1, ds2, ds3, ds4 = sorted_four

        # All 6 pairwise combos
        pairs = [
            (ds1, ds2), (ds1, ds3), (ds1, ds4),
            (ds2, ds3), (ds2, ds4), (ds3, ds4)
        ]
        dists = []
        for (x, y) in pairs:
            d_xy = pairwise_dict.get((p, x, y), float('nan'))
            dists.append(d_xy)

        valid_dists = [d for d in dists if not pd.isna(d)]
        if len(valid_dists) == 6:
            mean_dist = sum(valid_dists) / 6.0
        else:
            mean_dist = float('nan')

        quad_results.append({
            'Percent': p,
            'Datasets': f"{dsA},{dsB},{dsC},{dsD}",
            'Quad_Wasserstein': mean_dist,
            'PairwiseDistances': dists
        })

##########################################
# 6) Save results to CSV
##########################################
df_trip = pd.DataFrame(triplet_results)
df_quad = pd.DataFrame(quad_results)

trip_csv = os.path.join(base_dir, 'triplet_wasserstein.csv')
quad_csv = os.path.join(base_dir, 'quadruplet_wasserstein.csv')

df_trip.to_csv(trip_csv, index=False)
df_quad.to_csv(quad_csv, index=False)

print("Saved triplet distances to", trip_csv)
print("Saved quadruplet distances to", quad_csv)


# Weisfeiler-Lehman Kernel


In [ ]:
!pip install grakel

In [ ]:
import os
import dgl
import networkx as nx
import numpy as np
import pandas as pd

from grakel import Graph
from grakel import WeisfeilerLehman, VertexHistogram

###############################################
# 1) Converting NetworkX/DGL to Grakel format
###############################################
def dgl_to_grakel_graph(dgl_graph, label_key=None):
    """
    Converts a DGL graph to a Grakel-compatible graph object.
    Optionally uses 'label_key' to extract node labels from ndata
    if your nodes have attributes. Otherwise, default label=1.
    """
    nx_g = dgl.to_networkx(dgl_graph).to_undirected()

    # Grakel expects:
    # - A list of (node, neighbors) adjacency
    # - Optional node_labels dict: node_id -> label

    # Re-map nodes to 0..n-1 if not already
    mapping = {node: i for i, node in enumerate(nx_g.nodes())}
    nx_g = nx.relabel_nodes(nx_g, mapping)

    edges = list(nx_g.edges())
    # Convert edges to adjacency: a list of tuples (node1, node2)
    # No node labels? Let's just label them all '1'
    node_labels = {}
    for node in nx_g.nodes():
        node_labels[node] = 1

    # You could incorporate a real label if you have them in ndata:
    #   label = dgl_graph.ndata[label_key][node] if label_key else 1

    # Build a Grakel Graph
    grakel_graph = Graph(
        edges,
        node_labels=node_labels
    )
    return grakel_graph

###############################################
# 2) Example: Compare pairs of graphs with WL
###############################################
# Suppose you have pairs of datasets to compare
pairs_to_compare = [
    ('ml', 'virtualshakespeare'),
    ('algo004', 'comp'),
    ('ml', 'comp'),
    ('algo004', 'virtualshakespeare'),
    ('ml', 'algo004'),
    ('comp', 'virtualshakespeare')
]

percents = [0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75]
seeds = [18, 61, 53, 29, 69, 42, 2, 21, 78, 99]

base_dir = 'drive/MyDrive/Colab Notebooks/GNN Research/Graph Neural Network'
results = []

# We'll instantiate a Weisfeiler-Lehman + VertexHistogram pipeline
#  - 'n_iter=2' is the number of WL iterations (adjust as needed)
#  - VertexHistogram is a base kernel that just counts label frequencies
wl_kernel = WeisfeilerLehman(n_iter=2, base_graph_kernel=VertexHistogram)

for ds1, ds2 in pairs_to_compare:
    print(f"Comparing WL kernel for {ds1} vs. {ds2}")
    for p in percents:
        kernel_per_seed = []

        for seed in seeds:
            # Construct file paths
            ds1_path = os.path.join(
                base_dir, f'{ds1}_results', 'graphs', str(seed),
                f'graphs_{ds1}_{p}p_seed{seed}.bin'
            )
            ds2_path = os.path.join(
                base_dir, f'{ds2}_results', 'graphs', str(seed),
                f'graphs_{ds2}_{p}p_seed{seed}.bin'
            )

            if not os.path.isfile(ds1_path) or not os.path.isfile(ds2_path):
                print(f"Missing file for seed={seed}, p={p}, skipping...")
                continue

            # Load the DGL graphs
            g1_dgl, _ = dgl.load_graphs(ds1_path)
            g2_dgl, _ = dgl.load_graphs(ds2_path)
            # We assume the first graph in each bin is the relevant graph
            dgl1 = g1_dgl[0]
            dgl2 = g2_dgl[0]

            # Convert to Grakel format
            grakel_g1 = dgl_to_grakel_graph(dgl1)
            grakel_g2 = dgl_to_grakel_graph(dgl2)

            # The WL kernel in Grakel expects a list of graphs
            # We'll pass [g1, g2] and compute the full kernel matrix (2x2)
            # Then the off-diagonal element K[0,1] is the similarity
            g_list = [grakel_g1, grakel_g2]

            # Fit the kernel with these two graphs
            # 'fit_transform' yields a 2x2 kernel matrix
            K = wl_kernel.fit_transform(g_list)

            # The diagonal is K[0,0] = similarity(g1,g1), K[1,1] = similarity(g2,g2)
            # The cross-term is K[0,1] = similarity(g1,g2)
            similarity_g1_g2 = K[0,1]

            kernel_per_seed.append(similarity_g1_g2)

        # Average kernel similarity across seeds
        if len(kernel_per_seed) == 0:
            mean_kernel = np.nan
        else:
            mean_kernel = np.mean(kernel_per_seed)

        results.append({
            'Dataset1': ds1,
            'Dataset2': ds2,
            'Percent': p,
            'Mean_WL_Similarity': mean_kernel
        })

df = pd.DataFrame(results)
output_csv = os.path.join(base_dir, 'wl_kernel_comparison.csv')
df.to_csv(output_csv, index=False)
print(f"Saved WL kernel comparison to {output_csv}")


# KL Divergence

In [ ]:
import os
import dgl
import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import entropy

############################
# 1) Setup
############################
datasets = ['algo004', 'comp', 'ml', 'virtualshakespeare']
percents = [0.05, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75]
seeds = [18, 61, 53, 29, 69, 42, 2, 21, 78, 99]

base_dir = 'drive/MyDrive/Colab Notebooks/GNN Research/Graph Neural Network'

############################
# 2) Find global max degree
############################
global_max_degree = 0

for ds in datasets:
    for p in percents:
        for seed in seeds:
            folder_path = os.path.join(base_dir, f"{ds}_results", "graphs", str(seed))
            filename = f"graphs_{ds}_{p}p_seed{seed}.bin"
            filepath = os.path.join(folder_path, filename)

            if not os.path.isfile(filepath):
                continue

            graphs, _ = dgl.load_graphs(filepath)
            g_dgl = graphs[0]

            nx_g = dgl.to_networkx(g_dgl).to_undirected()
            if nx_g.number_of_nodes() == 0:
                continue

            degrees = [deg for _, deg in nx_g.degree()]
            if degrees:
                local_max = max(degrees)
                global_max_degree = max(global_max_degree, local_max)

print(f"Global max degree found: {global_max_degree}")

############################
# Helper to build distribution
############################
def build_fixed_distribution(degrees, max_deg):
    dist = np.zeros(max_deg + 1, dtype=float)
    for d in degrees:
        dist[d] += 1
    total = dist.sum()
    if total > 0:
        dist /= total
    return dist

############################
# 3) Collect/Average across seeds
############################
distributions = {}  # (dataset, percent) -> averaged distribution

for ds in datasets:
    for p in percents:
        dists_for_seeds = []
        for seed in seeds:
            folder_path = os.path.join(base_dir, f"{ds}_results", "graphs", str(seed))
            filename = f"graphs_{ds}_{p}p_seed{seed}.bin"
            filepath = os.path.join(folder_path, filename)

            if not os.path.isfile(filepath):
                continue

            graphs, _ = dgl.load_graphs(filepath)
            g_dgl = graphs[0]
            nx_g = dgl.to_networkx(g_dgl).to_undirected()

            if nx_g.number_of_nodes() == 0:
                continue

            degrees = [deg for _, deg in nx_g.degree()]
            dist = build_fixed_distribution(degrees, global_max_degree)
            dists_for_seeds.append(dist)

        # Average the distributions over seeds
        if len(dists_for_seeds) == 0:
            distributions[(ds, p)] = None
        else:
            mean_dist = np.mean(dists_for_seeds, axis=0)
            # Add small EPS to avoid zeros => infinite KL
            EPS = 1e-12
            mean_dist += EPS
            mean_dist /= mean_dist.sum()
            distributions[(ds, p)] = mean_dist

############################
# 4) Compute KL only for
#    ds1 != ds2 and p1 == p2
############################
rows = []
for p in percents:
    # We'll gather each dataset's distribution for this percent
    valid_ds = []
    for ds in datasets:
        dist = distributions.get((ds, p), None)
        if dist is not None:
            valid_ds.append(ds)

    # Now compare each pair of different datasets at this same percent
    for i in range(len(valid_ds)):
        for j in range(i+1, len(valid_ds)):
            ds1 = valid_ds[i]
            ds2 = valid_ds[j]
            # ds1 < ds2 or ds1 != ds2 check
            if ds1 == ds2:
                continue

            dist1 = distributions[(ds1, p)]
            dist2 = distributions[(ds2, p)]
            if dist1 is None or dist2 is None:
                continue

            kl_12 = entropy(dist1, dist2)
            kl_21 = entropy(dist2, dist1)
            kl_sym = 0.5 * (kl_12 + kl_21)

            rows.append({
                'Percent': p,
                'Dataset1': ds1,
                'Dataset2': ds2,
                'KL(P||Q)': kl_12,
                'KL(Q||P)': kl_21,
                'KL_sym': kl_sym
            })

# Sort results
df_kl = pd.DataFrame(rows)
df_kl.sort_values(by=['Percent','Dataset1','Dataset2'], inplace=True)
df_kl.reset_index(drop=True, inplace=True)

############################
# 5) Save to CSV
############################
out_csv = os.path.join(base_dir, "kl_divergence_same_percent.csv")
df_kl.to_csv(out_csv, index=False)
print(f"KL divergence results saved to {out_csv}")


### Source: Sizeof.ipynb

In [ ]:
import os
import numpy as np
import pandas as pd
import networkx as nx

datasets = ['algo004', 'comp', 'ml', 'virtualshakespeare']
base_path = '/content/drive/MyDrive/Colab Notebooks/GNN Research/SLN FL/data'

rows = []

for ds in datasets:
    path = os.path.join(base_path, f'w_removal_{ds}')
    df   = pd.read_csv(path, sep=" ", header=None,
                       names=["n1","n2","f1","f2","f3","f4","f5","f6","f7","l"])
    df.dropna(subset=['n1','n2'], inplace=True)

    # build node map
    nodes      = np.unique(df[['n1','n2']].values)
    idx_map    = {n:i for i,n in enumerate(nodes)}

    # filter only positive edges, drop self loops
    edge_df    = df[df.l==1]
    edge_df    = edge_df[edge_df.n1 != edge_df.n2]

    # map to integer indices
    edges      = [(idx_map[u], idx_map[v]) for u,v in edge_df[['n1','n2']].values]

    # build undirected graph
    G = nx.Graph()
    G.add_nodes_from(range(len(nodes)))
    G.add_edges_from(edges)

    # just in case: remove any self loops
    G.remove_edges_from(nx.selfloop_edges(G))

    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
    print(f"{ds:20s} → nodes: {n_nodes:5d}, edges: {n_edges:6d}")
    rows.append({'Dataset': ds, 'Nodes': n_nodes, 'Edges': n_edges})

# Save summary
summary = pd.DataFrame(rows)


In [ ]:
import os
import numpy as np
import pandas as pd

datasets  = ['algo004', 'comp', 'ml', 'virtualshakespeare']
base_path = '/content/drive/MyDrive/Colab Notebooks/GNN Research/SLN FL/data'

for ds in datasets:
    # 1) Load data
    path = os.path.join(base_path, f'w_removal_{ds}')
    df   = pd.read_csv(path, sep=" ", header=None,
                       names=["n1","n2","f1","f2","f3","f4","f5","f6","f7","l"])

    # — drop any rows with missing node IDs —
    df.dropna(subset=['n1','n2'], inplace=True)

    # 2) Keep only positive edges and drop self-loops
    df = df[df.l == 1]
    df = df[df.n1 != df.n2]

    # 3) Build node→index map
    nodes   = np.unique(df[['n1','n2']].values)
    idx_map = {node: idx for idx, node in enumerate(nodes)}
    n_nodes = len(nodes)

    # 4) Create adjacency matrix
    adj = np.zeros((n_nodes, n_nodes), dtype=int)
    for u, v in df[['n1','n2']].itertuples(index=False):
        ui, vi = idx_map[u], idx_map[v]
        adj[ui, vi] = 1
        adj[vi, ui] = 1

    # 5) Count only upper-triangle (no diagonal)
    n_edges = int(np.triu(adj, k=1).sum())

    print(f"{ds:20s} → Nodes = {n_nodes:5d},  Edges = {n_edges:6d}")


In [ ]:
import numpy as np
import os

def load_cm(path):
    return np.load(path)

def tpr_fpr(cm):
    tn, fp, fn, tp = cm.ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    return fpr, tpr

def compute_auc(fpr_list, tpr_list):
    sorted_points = sorted(zip(fpr_list, tpr_list))
    auc = 0.0
    for i in range(1, len(sorted_points)):
        x0, y0 = sorted_points[i-1]
        x1, y1 = sorted_points[i]
        auc += (x1 - x0) * (y0 + y1) / 2
    return auc

# Config
datasets = ['algo004', 'comp', 'ml', 'virtualshakespeare']
percents = [0.1, 0.25, 0.5, 0.75]
seeds = [18, 61, 53, 29, 69, 42, 2, 21, 78, 99]

for ds in datasets:
    print(f"\n===== Dataset: {ds} =====")
    base_path = f"drive/MyDrive/Colab Notebooks/GNN Research/Graph Neural Network/{ds}_results/cm_npy"
    for pct in percents:
        fprs = []
        tprs = []
        for seed in seeds:
            file_name = f"cm_{ds}_{pct}p_seed{seed}.npy"
            file_path = os.path.join(base_path, str(seed), file_name)
            if not os.path.exists(file_path):
                print(f"File not found: {file_path}")
                continue
            cm = load_cm(file_path)
            fpr, tpr = tpr_fpr(cm)
            fprs.append(fpr)
            tprs.append(tpr)
        if len(fprs) >= 2:
            auc = compute_auc(fprs, tprs)
            print(f"AUC at {int(pct * 100)}% = {auc:.4f}")
        else:
            print(f"Not enough data for AUC at {int(pct * 100)}%")


In [ ]:
!ls "drive/MyDrive/Colab Notebooks/GNN Research/Graph Neural Network/algo004_results/cm_npy/18/"